# 007 — LangGraph 사이드바 챗봇 (노트북 변수 인식)

JupyterLab **우측 사이드바**에 떠 있으면서 **현재 노트북 변수**를 보고 답하는 LangGraph 챗봇입니다.

- 노트북 변수는 커널의 네임스페이스에 살아있고, 이 코드는 커널 안에서 돌기 때문에 `get_ipython().user_ns` 로 직접 읽습니다.
- 로컬 LLM 은 표준 라이브러리 `urllib` 로만 연결 (openai SDK/httpx 미사용 — 폐쇄망 차단 패키지 회피).
- `sidecar` 가 없으면 현재 셀에 인라인으로 폴백합니다 (기능 동일).
- 추론 서버가 없어도 `MockLLM` 으로 변수 인식 동작을 바로 확인할 수 있습니다.

## 1. 사이드바 챗봇 열기 (MockLLM)

먼저 샘플 변수를 만들고 우측 사이드바 패널을 띄웁니다. 패널 상단의 **변수 패널** 에 아래 변수들이 보입니다.

In [ ]:
# 샘플 노트북 변수
x = 42
ratios = [0.1, 0.2, 0.7]
config = {'lr': 0.01, 'epochs': 10}
report_title = '2026 1분기 리포트'

from sidebar_chat import open_sidebar_chat
bot = open_sidebar_chat()   # MockLLM + 우측 사이드바 (sidecar 없으면 셀 인라인)
print('우측 사이드바(또는 아래 인라인 패널)에서 질문해 보세요. 예: 지금 어떤 변수가 보여?')

## 2. 변수를 더 만들고 새로고침

셀에서 새 변수를 만든 뒤 사이드바의 **변수 새로고침** 버튼을 누르면 챗봇이 인식합니다. (pandas/numpy 가 있으면 DataFrame/배열 요약도 확인)

In [ ]:
# (선택) pandas 가 있으면 DataFrame 요약도 인식되는지 확인
try:
    import pandas as pd
    sales = pd.DataFrame({'month': [1, 2, 3], 'amount': [100, 220, 180]})
except ImportError:
    sales = [100, 220, 180]   # pandas 없으면 리스트로 대체

customers = ['A사', 'B사', 'C사']
print('변수 생성 완료 → 사이드바에서 [변수 새로고침] 을 눌러보세요.')

## 3. 실제 사내 로컬 LLM 연결

`MockLLM` 대신 사내 OpenAI 호환 추론 서버(vLLM/Ollama/llama.cpp)에 붙입니다. 표준 라이브러리 `urllib` 만 사용하므로 추가 패키지가 필요 없습니다.

```python
from sidebar_chat import LocalOpenAILLM, open_sidebar_chat

llm = LocalOpenAILLM(
    base_url='http://localhost:8000/v1',  # 사내 추론 서버 주소
    model='qwen3',                          # 서버에 로드된 모델 이름
    # api_key='...',                        # 필요 시
)
bot = open_sidebar_chat(llm=llm)            # 값까지 보려면 include_values=True
```

## 4. UI 없이 프로그래매틱 사용 + HTML 내보내기

위젯 없이도 `chat()` / `export_html()` 로 쓸 수 있습니다. 대화 기록은 self-contained HTML 로 반출합니다.

In [ ]:
from sidebar_chat import SidebarChat

bot2 = SidebarChat()   # MockLLM
print(bot2.chat('내 노트북 변수들 요약해줘'))
print('---')
print('챗봇이 보는 변수:', [v['name'] for v in bot2.current_variables()])

path = bot2.export_html('conversation.html')   # 업무망 반출용 self-contained HTML
print('저장:', path)